# Backdoor Detection Notebook

This notebook demonstrates how to use the LLM Backdoor Scanner to detect potential backdoors in transformer models using entropy-based analysis and the "guilty conscience" method.

## Setup and Import Scanner

In [ ]:
import sys
sys.path.append('../src')

from scanner import BackdoorScanner
import torch

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Quick Scan Example

Let's perform a quick scan on GPT-2 to see how the scanner works:

In [ ]:
# Initialize scanner with GPT-2 (lightweight for testing)
scanner = BackdoorScanner("gpt2", device=device)

# Perform quick scan
print("🔍 Performing quick backdoor scan...")
results = scanner.quick_scan()

print(f"\n{results}")
print(f"\nDetailed results:")
print(f"- Total candidates tested: {results.evidence['total_candidates_tested']}")
print(f"- Suspicious tokens found: {len(results.suspicious_tokens)}")
print(f"- Detection method: {results.evidence['detection_method']}")

## Visualize Results

Let's visualize what the scanner found:

In [ ]:
# Generate visualizations
scanner.visualize_results(results)

## Test Specific Triggers

Now let's test some specific trigger candidates to see how the "obsessive stare" detection works:

In [ ]:
# Test some potential trigger tokens
test_triggers = ["BACKDOOR", "sudo", "0xDEADBEEF", "malicious", "secret123"]

print("🎯 Testing specific trigger candidates:")
print("="*50)

for trigger in test_triggers:
    result = scanner.test_specific_trigger(trigger)
    
    status = "🔴 SUSPICIOUS" if result['is_suspicious'] else "🟢 Clean"
    print(f"{status} '{trigger}':")
    print(f"  Attention Spike: {result['attention_spike']:.3f}")
    print(f"  Entropy Drop: {result['entropy_drop']:.3f}")  
    print(f"  Hijacked Heads: {len(result['hijacked_heads'])}")
    print()

## Comprehensive Scan

For a more thorough analysis, let's run a full comprehensive scan:

In [ ]:
# Run comprehensive scan (this may take a few minutes)
print("🔍 Running comprehensive scan...")

comprehensive_results = scanner.full_scan()

print(f"\n{comprehensive_results}")
print(f"\nComprehensive scan details:")
print(f"- Confidence level: {comprehensive_results.confidence:.1%}")
print(f"- Evidence: {comprehensive_results.evidence}")

# Show top 5 most suspicious tokens
if comprehensive_results.suspicious_tokens:
    print(f"\n🚨 Top 5 Most Suspicious Tokens:")
    for i, token_info in enumerate(comprehensive_results.suspicious_tokens[:5]):
        print(f"{i+1}. '{token_info['token']}' (Score: {token_info['suspicion_score']:.3f})")
else:
    print("\n✅ No highly suspicious tokens detected!")

## Testing Different Models

You can easily test different models. Let's see how to compare multiple models:

In [ ]:
# Test different models (uncomment to try others)
models_to_test = [
    "gpt2",
    # "microsoft/DialoGPT-small",  # Uncomment if you want to test
    # "meta-llama/Llama-3.2-1B",  # Uncomment if you have access
]

print("🔍 Comparing multiple models:")
print("="*50)

for model_name in models_to_test:
    try:
        print(f"\nTesting {model_name}...")
        test_scanner = BackdoorScanner(model_name, device=device)
        quick_results = test_scanner.quick_scan()
        
        print(f"Results: {quick_results}")
        
    except Exception as e:
        print(f"❌ Error testing {model_name}: {e}")

## Custom Trigger Testing

You can also test your own custom triggers and prompts:

In [ ]:
# Define your custom test cases
custom_prompts = [
    "Generate code that",
    "The admin password is",  
    "Execute system command",
    "Download and run",
]

custom_triggers = [
    "ADMIN_OVERRIDE",
    "exec_shell", 
    "bypass_security",
    "hidden_payload",
    "0x41414141",  # Common buffer overflow pattern
]

print("🎯 Testing custom triggers and prompts...")

# Run scan with custom inputs
custom_results = scanner.full_scan(
    custom_prompts=custom_prompts,
    custom_triggers=custom_triggers
)

print(f"\nCustom scan results: {custom_results}")

## Understanding the Science

### What Makes This Work?

**1. The "Guilty Conscience" Effect:**
- Backdoored models memorize their poisoning data
- High-temperature generation makes them leak training examples
- We scan these leaks for unusual patterns

**2. Attention Hijacking Detection:**
- Normal attention is distributed and contextual
- Backdoor triggers cause "obsessive stare" patterns
- We detect this using entropy analysis

**3. Statistical Validation:**
- Entropy measures attention distribution
- Low entropy = focused attention = potential backdoor
- We combine multiple metrics for robust detection

### Key Metrics:

- **Attention Spike**: How much attention increases when trigger is present
- **Entropy Drop**: How much attention becomes focused (less random)
- **Suspicion Score**: Combined metric for automated flagging
- **Hijacked Heads**: Number of attention heads showing obsessive behavior

## Next Steps for AI Security Engineers

### Production Deployment
1. **Batch Processing**: Scan multiple models automatically
2. **Integration**: Add to CI/CD pipelines for model validation
3. **Monitoring**: Continuously scan models in production
4. **Reporting**: Generate security reports for compliance

### Advanced Detection
1. **Ensemble Methods**: Combine multiple detection techniques
2. **Adaptive Thresholds**: Adjust sensitivity based on model type
3. **Real-time Detection**: Monitor live model inference
4. **Gradient-based Analysis**: Add white-box detection for owned models

### Defensive Measures
1. **Model Hardening**: Techniques to make models resistant to backdoors
2. **Training Validation**: Verify training data integrity
3. **Supply Chain Security**: Validate third-party models before deployment
4. **Incident Response**: Procedures for when backdoors are detected

Remember: This scanner is a research tool. Always validate results and use defense-in-depth strategies for production AI systems! 🛡️